# పాఠం 18 (తదుపరి): చర్యను ఒక *మానవుడు* అనుమతించినదనే సాక్ష్యాలు

పాఠం ఏమి చేయాలనుకున్నారో (**ఏజెంట్**) మరియు ఏమి నిర్ణయించారో (**గేట్**) సాక్ష్యం అందిస్తుంది. ఈ నోట్‌బుక్ ఆ కోల్పోయిన భాగాన్ని జోడిస్తుంది: ఒక **పేరుతో కూడిన మానవుడు** **ఖచ్చితమైన** చర్యకు అంగీకరించిన సాక్ష్యం — పూర్తి కానానికల్ చర్యపై ఒక వేరే, మానవుడి చేతి సంతకం, ఆఫ్‌లైన్ నిర్ధారణతో.

ఇక్కడ అందుబాటులో ఉన్న రెండు అబ్జెక్టులు పాఠంలో రిసిప్ట్ల వలే **అదే నమూనా ప్యాకేజీని** ఉపయోగిస్తాయి: ఒక ఫ్లాట్ పేలోడు, అందులో `type` అనే ఫీల్డ్ ఉంటుంది, కానానికల్ JCS బైట్లపై నేరుగా Ed25519 చేత సంతకం చేయబడింది, రెండర్ చేసిన `signature` ఆబ్జెక్ట్ జతచేయబడింది (మరియు సంతకం చేసిన బైట్లలో కాదు). అనుమతి రిసిప్ట్ ఒక కొత్త `type` (`human.approval.v1`), చర్య టైపు పక్కన ఉంది, కాబట్టి ఒకే `verify_chain` రెండూ కవర్ చేస్తుంది, మీ మెయిన్ నోట్‌బుక్ లోని కోడ్ మార్గం ద్వారా. ఈ మానవ-అనుమతి రిసిప్ట్ ఒక విద్యా సంయోజనంగా నిర్వచించబడింది, draft-farley-acta-signed-receipts ద్వారా నిర్వచించబడిన రిసిప్ట్ టైపు కాదు.

ప్రధాన నోట్‌బుక్ లోని డెమో వెరిఫయర్ కంటే deliberate గా ఒక అప్‌గ్రేడ్: ఇక్కడ వెరిఫయర్ `signature.key_id` ను సంతకం రిసిప్ట్ లో ఉన్న పబ్లిక్ కీ మీద విశ్వసించకుండా, **పిన్న్ చేసిన కీ రిజిస్ట్రీ** తో పరిష్కరిస్తుంది. ఇదే ఆ పాఠంలోని తన స్వంత చెక్‌లిస్ట్ సూచించే ఉత్పత్తి స్థితి ("ప్రూఫ్ పబ్లిక్ కీ ప్రచురించు"), ఇది మోసపూరిత చర్యను అంగీకరించకుండా, మీదుగా ఒక కీ తీసుకురావడం కాదు.

ఈ నోట్‌బుక్ నేర్పించే నియమం: **ఒక సంతకం చేసిన అనుమతి మాత్రం అధికారంగా ఉండదు.** అధికారమేమిటంటే, అనుమతి రిసిప్ట్ మరియు చర్య రిసిప్ట్ ఎక్సిక్యూషన్ సమయానికి ఒకే కానానికల్ చర్యను కట్టుబడి ఉండాలి, ఇప్పటికీ చెలామణిలో ఉన్న విధాన వెర్షన్, కీ, గడువు పైన ఉండాలి, మరియు ఇంకేమైనా వినియోగించబడకపోవాలి. ప్రతి విఫలతకు ఒక **వేరే కారణం** ఉంటుంది, కాబట్టి మీరు *అధికారము పాతదిగా మారింది* మరియు *నేర్చిన చర్య మారిందా* వేరు చేసుకోగలరు.


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## ఖచ్చితమైన చర్య

అనుమతికి ఉపయోగించే యూనిట్ **క్యానానికల్ చర్య వస్తువు** — "రీఫండ్ అంగీకరించు" వంటి అస్పష్టమైన లేబుల్ కాదు, కానీ ఖచ్చితంగా, పూర్తిగా నిర్వచించబడిన చర్య. మొత్తం వస్తువుపై సంతకం చేయడం (మరియు అందునుంచి డైజెస్ట్ పొందడం) మనకి తరువాత మనుష్యులు *ఈ*ని మాత్రమే అనుమతించారని నిరూపించడానికి మార్గం ఇస్తుంది.


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## ఒక ఇన్వెలప్, రెండు అధికారాలు

ప్రతి రసీదు పాఠం యొక్క ఇన్వెలప్: ఒక రేఖీయ ప్రయోగం `type` ఫీల్డ్‌తో, మరియు `signature` ఆబ్జెక్ట్‌తో (`alg`, `sig`, `key_id`) ఉంటుంది, ఇది సంతకం చేసిన బైట్ల భాగం కాదు. `verify_envelope` అనేది రెండు రసీదు రకాల కోసం పొందుబడే నిర్మాణాత్మక + సంతకం తనిఖీ; ఇది ఏ **పిన్నడ్ కీ రిజిస్ట్రీ** `signature.key_id` కు అన్వయిస్తుంది అనేది అధికారాలను వేరుగా ఉంచుతుంది:

- **ఆమోద రసీదు** (`human.approval.v1`) — పేరుగల ఆమోదదారు, పూర్తి కెనోనికల్ క్రియ మరియు దాని డైజెస్ట్, `policy_version`, ఇష్యూ + గడువు సమయాలు. ఒకసారి వినియోగం చైన్ స్థాయిలో ట్రాక్ చేయబడుతుంది.
- **క్రియ రసీదు** (`agent.action.v1`) — ఏజెంట్ గుర్తింపు, `run_id`, అదే కెనోనికల్ క్రియ డైజెస్ట్, అమలు ఫలితం + సమయం, మరియు `parent_approval_ref`: ఆమోదం యొక్క `receipt_hash`, పాఠం యొక్క చైన్‌లోని `previous_receipt_hash` వలెనే కన్వెన్షన్.

భాగస్వామ్య `action_digest` ఫీల్డ్ అనేది బైండింగ్ ఆధారపడే జత. `key_id` సంతకం ఆబ్జెక్టులో ఒక సూచన మాత్రమే గా ఉంటుంది: దాన్ని వేరే పిన్నడ్ కీలోకి మళ్ళీ చూపిస్తే సంతకం తనిఖీ విఫలమవుతుంది, అందుచేత ఇది ఏమి ఇస్తుంది కాదు.


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the canonical JCS
    bytes directly; the signature object is NOT part of the signed bytes."""
    canonical = canonicalize(payload)
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(canonical).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized and passed directly to Ed25519.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        canonical = canonicalize(payload)
        VerifyKey(b64url_decode(pub)).verify(canonical, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: బైండింగ్ నిజంగా నిర్ణయించే స్థానము  

`verify_chain` రెండు సంతకం తనిఖీలపై సౌకర్య ర్యాపర్ కాదు. ఇది ఒక చోటే భాగస్వామ్యమైన సాధారణ `action_digest`, ఆమోదం యొక్క విధాన/కీ/కాలపరిమితి **తాజాదనం**, మరియు ఆమోదం యొక్క **ఒకసారిగా వినియోగం**ను ఒకచోటా తనిఖీ చేస్తుంది, ఆ చర్య *ఈ సమయంలో* అమలు అవుతుందో లేదో.  

ప్రతి విఫలతకు **విభిన్న కారణం** ఉంటుంది, అందువల్ల తిరస్కరణను చదవునవాడు అధికారముని పాతదిగా మారిందా (నీతులు మార్చబడినవా, కీ మార్పు, ఆమోదం కాలం తేలిపోయిందా, ఆమోదం వినియోగించబడిందా) లేదా అమలులో ఉన్న చర్య ఇంకా చెల్లుబాటు అయ్యే ఆమోదం కింద నుండి మారిందా (డైజెస్ట్ మార్పు) అని తెలుసుకోగలడు.  


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## బైండింగ్ ఏం పట్టుకుంటుంది

కింద ప్రతి కేసు **వేరుగా కారణం** తో ** మూసివేయబడుతుంది**. మొదటి బ్లాక్ క్లాసిక్ సెట్ (తప్పడం, క్లిష్ట ప్రతినిధి, మళ్ళీ ప్లే, అధికారం మీద జాలం, పాడు ఇన్‌పుట్) ఉంది. రెండవ బ్లాక్ ఆ జంట, ప్రాపర్టీని నిర్ధారించడమే కాక, నయంగా మార్చుతుంది:

- **పాత అధికారం** — సంతకం ఇంకా చెల్లుబాటు ..., కానీ విధాన సంచిక మార్చబడింది, ఆమోదదారు కీ పిన్ చేసిన రిజిస్ట్రి నుండి తొలగించబడింది, లేదా ఆమోదం అమలుకు ముందు గడువు ముగిసింది;
- **డైజెస్ట్ మార్చడం** — చెల్లుబాటు సంతకం చేసిన చర్య రసీదు `parent_approval_ref` ఒక *ప్రామాణిక* ఆమోదాన్ని సూచిస్తుంది, కానీ ఆ ఆమోదపు కేనానికల్ చర్య డైజెస్ట్ నిజంగా అమలులో ఉన్న చర్యకు సరిపోలదు.


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## ఇది ఏం నిరూపిస్తుంది — మరియు ఇది ఏం నిరూపించదు

**నిరూపిస్తోంది:** ఒక పేరు ఉన్న మానవుడు *ఈ ఖచ్చితమైన కానానికల్ క్రియను* ఆమోదించిందని (పూర్తి క్రియ + డైజెస్ట్, పిన్ చేయబడిన రిజిస్ట్రీ నుండి పరిష్కరించిన కీ తో సంతకం), మరియు ఏజెంట్ *ఖచ్చితమైన ఆ ఆమోదించబడిన క్రియను* నిర్వహించింది (అనేనే డైజెస్ట్, ఆమోదానికి `receipt_hash` ద్వారా సరిగా బంధించిన రసీదుతో, పాఠం యొక్క స్వంత చైన్ సంప్రదాయం) — ఆమోదన విధానం యొక్క వెర్షన్, కీ, మరియు గడువు ఇంకా చెలామణిలో ఉన్నప్పుడు, కేవలం ఒకసారి. ఏదైనా వైపు మారితే, చైన్ మూసివేస్తుంది, మరియు తిరస్కరణ కారణం మీకు **ఏది** ఆస్తి విరిగిందో చెబుతుంది: పాత అధికారంతో మారిన చర్య.

**నిరూపించదు:** ఆమోద UI మానవుడికి వారు సంతకం చేస్తున్నది ఏమిటో చూపించిందని (WYSIWYS స్వంత సమస్య), రోటేషన్ తర్వాత కీ బలవంతంగా తీసుకోబడిందో లేదా దొంగలించబడిందో లేదో, లేదా ద్‍రుష్టతల ప్రభావాలు ఆ చర్యకు సరిపోయాయో కాదో. సంతకం అంటే ఆమోదం కాదు: పాత విధానంపై సరైన సంతకం, మారిన కీ, గడువు ముగిసిన విండో, లేదా భిన్నమైన డైజెస్ట్ ఇక్కడ ఏమీ ఇస్తాయి కాదు.

రెండు రసీదు రకాలు పాఠం యొక్క ఇన్వెలప్ మరియు ఒకే `verify_chain` కోడ్ మార్గం పంచుకుంటాయి ఉద్దేశపూర్వకంగా: మీరు ప్రధాన నోట్బుక్ లో చర్య రసీదు కోసం తయారు చేసిన బంధనమే మానవ ఆమోదాన్ని తనిఖీ చేసే కోడ్. ఒక వెరిఫెయిర్ ఒప్పందం, వేర్వేరు పిన్ చేయబడిన అధికారాలు, కానానికల్ చర్య డైజెస్ట్ తో కలసి మరియు మరేదీ కాదు.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**అస్వీకరణ**:
ఈ పత్రం AI అనువాద సేవ [Co-op Translator](https://github.com/Azure/co-op-translator) ఉపయోగించి అనువదించబడింది. మేము ఖచ్చితత్వానికి ప్రయత్నిస్తున్నప్పటికీ, ఆటోమేటెడ్ అనువాదాలు తప్పులు లేదా అసమగ్రతలను కలిగి ఉండవచ్చు. దాని స్వదేశ భాషలో ఉన్న అసలు పత్రాన్ని అధికారం కలిగిన మూలంగా పరిగణించాలి. కీలకమైన సమాచారం కోసం, ప్రొఫెషనల్ మానవ అనువాదాన్ని సిఫారసు చేస్తాము. ఈ అనువాదం ఉపయోగం వల్ల కలిగే ఏవైనా అపార్థాలు లేదా తప్పుదారులు కోసం మేము బాధ్యత వహించము.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
